**CELL 1**

In [1]:
import os, random, math, re, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Device:", device)


Torch: 2.6.0+cu124 | CUDA: True | Device: cuda


**CELL 2**

In [2]:
# ==== PREPROCESS CONFIG ====
# Đường dẫn CSV trên Kaggle (sửa cho đúng dataset của bạn)
CSV_PATH = "/kaggle/input/300-seconds/machine_usage_days_1_to_8_grouped_300_seconds.csv"

# Nếu biết tên cột demand (ví dụ 'cpu_util_percent') thì điền:
DEMAND_COL_NAME = None
# Hoặc dùng index (ưu tiên tên hơn index):
DEMAND_COL_INDEX = None

# Nếu có cột thời gian:
TIMESTAMP_CANDIDATES = ["timestamp","time","datetime","ts"]

# Có cần resample về 300s không? (để False nếu CSV đã grouped 300s)
RESAMPLE_TO_300S = False

# Chuẩn hoá z-score?
APPLY_SCALING = True

# Số cột "covariates" ở CUỐI dataframe (nếu có, ví dụ đặc trưng thời gian)
# Nếu không rõ thì để 0 cho đơn giản
COV_DIM = 0


**CELL 3**

In [3]:
assert os.path.exists(CSV_PATH), f"Không tìm thấy CSV: {CSV_PATH}"
df_raw = pd.read_csv(CSV_PATH)
df = df_raw.copy()
print("Raw df:", df.shape)

# Xử lý timestamp (nếu có)
ts_col = None
for c in TIMESTAMP_CANDIDATES:
    if c in df.columns:
        ts_col = c; break
if ts_col is not None:
    df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")
    df = df.sort_values(ts_col).reset_index(drop=True)
    if RESAMPLE_TO_300S:
        df = df.set_index(ts_col).resample("300S").mean().reset_index()

# Chọn cột demand
if DEMAND_COL_NAME is not None and DEMAND_COL_NAME in df.columns:
    demand_name = DEMAND_COL_NAME
elif DEMAND_COL_INDEX is not None and 0 <= int(DEMAND_COL_INDEX) < df.shape[1]:
    demand_name = df.columns[int(DEMAND_COL_INDEX)]
else:
    lower = {c.lower(): c for c in df.columns}
    for k in ["cpu","cpu_usage","cpu_util_percent","usage","demand","load","util","utilization"]:
        if k in lower: demand_name = lower[k]; break
    else:
        demand_name = df.columns[0]

# Đưa demand về cột 0 (bỏ cột timestamp nếu có)
feature_cols = [c for c in df.columns if c != demand_name and c not in TIMESTAMP_CANDIDATES]
ordered_cols = [demand_name] + feature_cols
df = df[ordered_cols].copy()

# Missing
df[ordered_cols] = df[ordered_cols].interpolate(method="linear", limit_direction="both")
df[ordered_cols] = df[ordered_cols].ffill().bfill()

# Scaling
if APPLY_SCALING:
    means = df[ordered_cols].mean(0)
    stds  = df[ordered_cols].std(0).replace(0, 1e-6)
    df[ordered_cols] = (df[ordered_cols]-means)/stds
    SCALER = {"means": means.to_dict(), "stds": stds.to_dict(), "cols": ordered_cols}
else:
    SCALER = None

data_np = df.values.astype(np.float32)
DEMAND_COL_IDX = 0
DX_TOTAL = data_np.shape[1]
X_DIM = DX_TOTAL - COV_DIM     # số feature cho past_x
print("Processed df:", df.shape, "| X_DIM:", X_DIM, "| COV_DIM:", COV_DIM, "| DEMAND_COL_IDX:", DEMAND_COL_IDX)


Raw df: (2243, 5)
Processed df: (2243, 5) | X_DIM: 5 | COV_DIM: 0 | DEMAND_COL_IDX: 0


**CELL 4**

In [4]:
# === Cell #4: DAPMForecaster (probabilistic) ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class TrendBlock(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim, num_layers=2):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.GELU()]
        for _ in range(num_layers - 1):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.GELU()])
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class DAPMForecaster(nn.Module):
    """
    forward(past_x, past_u, future_u) -> (mu, logvar)  [B, H, out_dim]
    """
    def __init__(self, feature_dim, covariate_dim, hidden_dim, output_dim, horizon,
                 n_heads=4, n_lstm_layers=2, dropout=0.2):
        super().__init__()
        self.horizon = horizon
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.feature_dim = feature_dim

        # Seasonal branch
        self.season_feature_embed = nn.Linear(feature_dim + covariate_dim, hidden_dim)
        self.season_query_embed   = nn.Linear(covariate_dim, hidden_dim)
        self.activation = nn.GELU()
        self.lstm_encoder = nn.LSTM(
            input_size=hidden_dim, hidden_size=hidden_dim,
            num_layers=n_lstm_layers, batch_first=True,
            dropout=dropout if n_lstm_layers > 1 else 0.0
        )
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=n_heads, batch_first=True, dropout=dropout
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        # Probabilistic decoder: outputs (mu, logvar)
        self.season_decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, output_dim * 2)   # -> (mu, logvar)
        )
        # Trend branch
        self.trend_encoder = nn.Sequential(
            nn.Linear(288 * feature_dim, hidden_dim), nn.GELU()
        )
        self.trend_decoder = TrendBlock(hidden_dim, horizon * output_dim, hidden_dim)

    def forward(self, past_x, past_u, future_u):
        """
        past_x: [B, L, X_DIM]  (z-score scale)
        past_u: [B, L, COV_DIM] or empty
        future_u: [B, H, COV_DIM] or empty
        returns: (mu, logvar) each [B, H, out_dim]
        """
        B = past_x.shape[0]
        last_value = past_x[:, -1:, 0:self.output_dim]  # anchor (B,1,out)

        # --- Seasonal ---
        if past_u is None or past_u.shape[-1] == 0:
            past_u = torch.zeros(past_x.shape[0], past_x.shape[1], 0, dtype=past_x.dtype, device=past_x.device)
        if future_u is None or future_u.shape[-1] == 0:
            future_u = torch.zeros(past_x.shape[0], self.horizon, 0, dtype=past_x.dtype, device=past_x.device)

        past_input_season = torch.cat([past_x, past_u], dim=-1)
        past_embed_season = self.activation(self.season_feature_embed(past_input_season))
        enc_out, _ = self.lstm_encoder(past_embed_season)
        query = self.activation(self.season_query_embed(future_u))
        attn_out, _ = self.attention(query, enc_out, enc_out)
        attn_out = self.norm1(query + attn_out)
        season_raw = self.season_decoder(attn_out)                    # (B,H,2*out)
        mu_s, lv_s = torch.chunk(season_raw, 2, dim=-1)               # log-variance

        # --- Trend ---
        trend_input = past_x.reshape(B, -1)
        trend_encoded = self.trend_encoder(trend_input)
        trend_mu_flat = self.trend_decoder(trend_encoded)             # (B,H*out)
        mu_t = trend_mu_flat.reshape(B, self.horizon, self.output_dim)
        lv_t = torch.zeros_like(mu_t)                                  # nếu muốn, có thể học var cho trend

        # --- Combine + anchor ---
        mu = last_value + mu_s + mu_t
        # tổng phương sai (giả định độc lập): var = var_s + var_t
        var = lv_s.exp() + lv_t.exp() + 1e-6
        logvar = torch.log(var)
        return mu, logvar

    @torch.no_grad()
    def predict(self, past_x, past_u, future_u, device=None):
        """
        NumPy inputs -> (mean, std) in the same (z-score) scale, shape (H, out_dim)
        """
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        if isinstance(past_x, np.ndarray):
            past_x = torch.tensor(past_x, dtype=torch.float32, device=device).unsqueeze(0)
            past_u = torch.tensor(past_u, dtype=torch.float32, device=device).unsqueeze(0) if past_u is not None else None
            future_u = torch.tensor(future_u, dtype=torch.float32, device=device).unsqueeze(0) if future_u is not None else None
        self.eval()
        mu, lv = self.forward(past_x, past_u, future_u)
        mu = mu[0].detach().cpu().numpy()
        std = (lv[0].detach().cpu().exp().sqrt().numpy())
        return mu, std


**CELL 5**

In [5]:
# === Cell #5: CPUForecaster (probabilistic, demand_col=0) ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class CPUForecaster(nn.Module):
    """
    forward(past_x, past_cov, future_cov) -> (mu, logvar)  [B, H, 1]
    - Neo đúng cột demand qua demand_col (mặc định 0)
    - Có variance head + tham số log_std_scale để hiệu chỉnh nhẹ
    """
    def __init__(self, x_dim, cov_dim, y_dim, hidden_dim, horizon,
                 num_heads=8, n_lstm_layers=2, dropout=0.2, demand_col: int = 0):
        super().__init__()
        self.horizon = horizon
        self.demand_col = int(demand_col)

        self.encoder_embedding = nn.Linear(x_dim + cov_dim, hidden_dim)
        self.lstm_encoder = nn.LSTM(
            input_size=hidden_dim, hidden_size=hidden_dim,
            num_layers=n_lstm_layers, batch_first=True,
            dropout=dropout
        )
        self.query_encoder = nn.Linear(cov_dim, hidden_dim)
        self.attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)

        # mean head
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2), nn.ReLU(),
            nn.Linear(hidden_dim * 2, y_dim)
        )
        # log-variance head
        self.var_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, y_dim)
        )
        self.log_std_scale = nn.Parameter(torch.zeros(1))  # learnable calibration

    def forward(self, past_x, past_cov, future_cov):
        # neo điểm cuối cột demand
        last_value = past_x[:, -1:, [self.demand_col]]  # (B,1,1)

        past_input = torch.cat([past_x, past_cov], dim=-1) if (past_cov is not None and past_cov.shape[-1] > 0) else past_x
        past_embed = F.relu(self.encoder_embedding(past_input))
        lstm_out, _ = self.lstm_encoder(past_embed)

        query = self.query_encoder(future_cov) if (future_cov is not None and future_cov.shape[-1] > 0) else torch.zeros(past_x.size(0), self.horizon, self.encoder_embedding.out_features, device=past_x.device)
        attn_out, _ = self.attention(query, lstm_out, lstm_out)
        z = self.norm(attn_out)

        delta = self.decoder(z)
        mu = last_value + delta
        logvar = self.var_head(z) + self.log_std_scale
        return mu, logvar

    @torch.no_grad()
    def predict(self, past_x, past_cov, future_cov, device=None):
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        if isinstance(past_x, np.ndarray):
            past_x = torch.tensor(past_x, dtype=torch.float32, device=device).unsqueeze(0)
            past_cov = torch.tensor(past_cov, dtype=torch.float32, device=device).unsqueeze(0) if past_cov is not None else None
            future_cov = torch.tensor(future_cov, dtype=torch.float32, device=device).unsqueeze(0) if future_cov is not None else None
        self.eval()
        mu, lv = self.forward(past_x, past_cov, future_cov)
        mu = mu[0].detach().cpu().numpy()
        std = (lv[0].detach().cpu().exp().sqrt().numpy())
        # tuỳ chọn: scale thêm nếu bạn đã set anp_model.pred_std_scale
        s = float(getattr(self, "pred_std_scale", 1.0))
        return {"mean": mu, "std": std * s}


**CELL 6**

In [6]:
# BẮT BUỘC: 2 class phải có trong scope
assert 'DAPMForecaster' in globals(), "Thiếu class DAPMForecaster (dán class ở Cell 4)."
assert 'CPUForecaster'  in globals(), "Thiếu class CPUForecaster (dán class ở Cell 4)."

# Dự báo 1 bước cho RL (env chỉ cần 1-step)
HORIZON = 1
HIDDEN  = 128

dapm_model = DAPMForecaster(
    feature_dim=X_DIM,
    covariate_dim=COV_DIM,
    hidden_dim=HIDDEN,
    output_dim=1,     # 1 mục tiêu (demand/cpu)
    horizon=HORIZON
).to(device)

anp_model  = CPUForecaster(
    x_dim=X_DIM,
    cov_dim=COV_DIM,
    y_dim=1,
    hidden_dim=HIDDEN,
    horizon=HORIZON,
    demand_col=0
).to(device)

# End-to-end: mở gradient
for p in dapm_model.parameters(): p.requires_grad = True
for p in anp_model.parameters():  p.requires_grad = True

print("Models ready:", type(dapm_model).__name__, "&", type(anp_model).__name__)


/usr/local/lib/python3.11/dist-packages/torch/nn/init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


Models ready: DAPMForecaster & CPUForecaster


**CELL 7**

In [7]:
# === Cell #7: TaskSampler (choices updated) ===
from dataclasses import dataclass
from typing import Tuple, Dict, List, Optional
import numpy as np

@dataclass
class TaskConfig:
    sla_cpu: float      # ví dụ 0.7
    vm_min: int
    vm_max: int
    cost_per_vm: float
    spike_amp: float
    window: Tuple[int, int]

class TaskSampler:
    def __init__(self, day_windows: Dict[int, Tuple[int,int]], seed: int = 123):
        self.day_windows = day_windows
        self.rng = np.random.default_rng(seed)
        self.days = list(day_windows.keys())

    def sample_one(self) -> TaskConfig:
        d = int(self.rng.choice(self.days))
        return TaskConfig(
            sla_cpu=float(self.rng.choice([0.6, 0.7, 0.8])),
            vm_min=int(self.rng.choice([1, 2, 3])),
            vm_max=int(self.rng.choice([8, 10, 12, 16])),
            cost_per_vm=float(self.rng.choice([0.2, 0.3, 0.4])),
            spike_amp=float(self.rng.choice([1.0, 1.2, 1.4, 1.6])),
            window=self.day_windows[d]
        )

    def sample(self, n: int) -> List[TaskConfig]:
        return [self.sample_one() for _ in range(n)]


**CELL 8**

In [8]:
# === Cell 8: forecast_infer_no_grad + RealEnv (FULL; adaptive SLA & risk) ===
import numpy as np
import torch

@torch.no_grad()
def forecast_infer_no_grad(past_x, past_u, fut_u, horizon=1, device=None):
    """
    Gọi DAPM & ANP (bản của bạn), trộn mean/std (z-score scale).
    Trả: mean, std shape (H,1).
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    # DAPM
    mu_d, sd_d = dapm_model.predict(past_x, past_u, fut_u, device=device)  # (H,1),(H,1)
    # ANP
    out = anp_model.predict(past_x, past_u, fut_u, device=device)          # {"mean":(H,1), "std":(H,1)}
    mu_a, sd_a = out["mean"], out["std"]
    # mix
    mean = 0.5 * (mu_d + mu_a)
    var  = 0.5 * (sd_d**2 + sd_a**2) + 1e-6
    std  = np.sqrt(var)
    H = int(horizon)
    return mean[:H], std[:H]


class RealEnv:
    """
    Env thật-giản lược: dùng forecast -> scale VM -> reward risk-aware.
    Quan sát 9 chiều: [last_d, mean, std, gap, risk95, vm, sla, prev_action, prev_reward].
    """
    def __init__(self, data_np, task, max_steps=128, history_len=288,
                 demand_col=0, cov_dim=0, rng=None):
        self.data        = data_np
        self.tcfg        = task
        self.max_steps   = max_steps
        self.history_len = history_len
        self.demand_col  = demand_col
        self.cov_dim     = cov_dim
        self.rng         = rng or np.random.default_rng(123)

        self.idx = None
        self.vm  = None
        self.prev_action = 1
        self.prev_reward = 0.0
        self.reset()

    # ---------- helpers ----------
    def _inv_scale_vec(self, z_vec):
        if 'SCALER' in globals() and SCALER is not None:
            col0 = list(SCALER["means"].keys())[0]
            mu   = SCALER["means"][col0]
            sd   = SCALER["stds"][col0] if SCALER["stds"][col0] != 0 else 1e-6
            return z_vec * sd + mu
        return z_vec

    def _get_demand(self, idx:int) -> float:
        z = float(self.data[idx, self.demand_col])
        real = float(self._inv_scale_vec(np.array([z], np.float32))[0])
        real = float(np.clip(real, 0.0, 100.0))
        return real * float(self.tcfg.spike_amp)

    def _split_past_cov(self, idx):
        hist = np.arange(idx - self.history_len, idx)
        past_all = self.data[hist, :]
        if self.cov_dim > 0:
            past_x = past_all[:, :-self.cov_dim]
            past_u = past_all[:, -self.cov_dim:]
            fut_u  = self.data[idx:idx+1, -self.cov_dim:]
        else:
            past_x = past_all
            past_u = np.zeros((self.history_len, 0), np.float32)
            fut_u  = np.zeros((1, 0), np.float32)
        return past_x, past_u, fut_u

    # ---------- gym-like ----------
    def reset(self):
        s, e = self.tcfg.window
        # Guard: nếu window ngắn, kẹp chỉ số an toàn
        low  = s + self.history_len + 1
        high = e - 1
        if high <= low:
            self.idx = max(s + self.history_len, min(high, low))
        else:
            self.idx = int(self.rng.integers(low, high))
        self.step_count = 0
        self.vm = int(self.tcfg.vm_min)
        self.prev_action = 1
        self.prev_reward = 0.0
        return self._make_obs()

    def _make_obs(self):
        past_x, past_u, fut_u = self._split_past_cov(self.idx)
        mean, std = forecast_infer_no_grad(past_x, past_u, fut_u, horizon=1)
        last_d = past_x[-1, self.demand_col].reshape(1)                 # z-score
        gap    = (mean.reshape(-1) - last_d).astype(np.float32)

        target_pct = self.tcfg.sla_cpu * 100.0
        # risk95 (%CPU)
        if 'SCALER' in globals() and SCALER is not None:
            col0 = list(SCALER["means"].keys())[0]
            mu0  = SCALER["means"][col0]
            sd0  = SCALER["stds"][col0] if SCALER["stds"][col0] != 0 else 1e-6
            risk95 = ((mean.reshape(-1)[0] + 1.64*std.reshape(-1)[0]) * sd0 + mu0) - target_pct
        else:
            risk95 = (mean.reshape(-1)[0] + 1.64*std.reshape(-1)[0]) - target_pct
        risk95 = np.array([risk95], dtype=np.float32)

        feats = [
            last_d.astype(np.float32),
            mean.reshape(-1).astype(np.float32),
            std.reshape(-1).astype(np.float32),
            gap,
            risk95,
            np.array([self.vm, self.tcfg.sla_cpu], np.float32),
            np.array([self.prev_action, self.prev_reward], np.float32),
        ]
        return np.concatenate(feats, axis=0).astype(np.float32)

    def step(self, action:int):
        action = int(np.clip(action, 0, 2))

        # scale VM
        if action == 0 and self.vm > self.tcfg.vm_min: self.vm -= 1
        elif action == 2 and self.vm < self.tcfg.vm_max: self.vm += 1

        # hệ thống thực
        demand   = self._get_demand(self.idx)
        noise    = self.rng.uniform(-1.0, 1.0)
        cpu_util = np.clip(demand / max(1, self.vm) + noise, 0.0, 100.0)

        # ----- Adaptive tolerance & risk scaling (FULL; siết hơn) -----
        # volatility của 96 bước cuối (đơn vị %CPU)
        hist = np.arange(self.idx - min(self.history_len, 192), self.idx)
        past_z = self.data[hist, self.demand_col]
        past_real = self._inv_scale_vec(past_z)
        vol = float(np.std(past_real[-96:])) if past_real.size > 0 else 0.0

        target_pct = self.tcfg.sla_cpu * 100.0
        err_pct = cpu_util - target_pct

        # Dung sai siết nhẹ: base 7.5 (trước 8.0)
        base_tol = 7.5
        tol_pct  = float(np.clip(base_tol + 0.05 * vol, 7.0, 11.0))
        inside   = 1.0 - min(abs(err_pct) / tol_pct, 1.0)  # [0..1]

        # Phạt vượt SLA mạnh hơn (-7.0), thiếu nhẹ
        r_sla_pos = 3.0 * inside
        r_over    = -7.0 * max(0.0,  err_pct) / 100.0
        r_under   = -1.0 * max(0.0, -err_pct) / 100.0
        sla_term  = r_sla_pos + r_over + r_under

        # Chi phí: tăng nhẹ gamma 0.18 để đỡ vung VM
        rel_vm     = max(0, self.vm - self.tcfg.vm_min)
        cost_gamma = 0.18
        cost_term  = - cost_gamma * rel_vm

        # Risk đuôi (q99): scale mạnh hơn theo volatility
        past_x, past_u, fut_u = self._split_past_cov(self.idx)
        mean, std = forecast_infer_no_grad(past_x, past_u, fut_u, horizon=1)
        if 'SCALER' in globals() and SCALER is not None:
            col0 = list(SCALER["means"].keys())[0]
            mu0  = SCALER["means"][col0]
            sd0  = SCALER["stds"][col0] if SCALER["stds"][col0] != 0 else 1e-6
            q99  = (mean.reshape(-1)[0] + 2.33*std.reshape(-1)[0]) * sd0 + mu0
        else:
            q99  = mean.reshape(-1)[0] + 2.33*std.reshape(-1)[0]
        risk_scale = 2.2 + 0.015 * vol
        risk_pen   = - risk_scale * max(0.0, (q99 - target_pct) / 100.0)

        # Hạn chế lắc hành động
        switch_pen = -0.1 if (action != self.prev_action) else 0.0

        reward = float(sla_term + cost_term + risk_pen + switch_pen)

        # advance
        self.idx += 1
        self.step_count += 1
        done = (self.step_count >= self.max_steps)
        self.prev_action = action
        self.prev_reward = reward

        info = {"vm": self.vm, "cpu_util": cpu_util, "tol_pct": tol_pct, "vol": vol}
        return self._make_obs(), reward, done, info


**CELL 9**

In [9]:
# === Cell 9: Policy & PPO (FULL; ObsNorm + KL early stop + epochs=10) ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class ObsNormalizer:
    def __init__(self, eps=1e-8):
        self.mean = None
        self.var  = None
        self.count= eps
    def update(self, x_np):
        x = np.asarray(x_np, dtype=np.float64)
        if self.mean is None:
            self.mean = x.mean(0); self.var = x.var(0) + 1e-6; self.count = x.shape[0]
        else:
            bm, bv, bc = x.mean(0), x.var(0) + 1e-6, x.shape[0]
            delta = bm - self.mean; tot = self.count + bc
            new_mean = self.mean + delta * bc / tot
            m_a = self.var * self.count; m_b = bv * bc
            M2 = m_a + m_b + delta**2 * self.count * bc / tot
            self.mean = new_mean; self.var = M2 / tot; self.count = tot
    def transform(self, x_t):
        if self.mean is None: return x_t
        mean = torch.tensor(self.mean, dtype=x_t.dtype, device=x_t.device)
        std  = torch.tensor(np.sqrt(self.var), dtype=x_t.dtype, device=x_t.device)
        if x_t.dim()==3:  # [B,1,obs]
            return (x_t - mean.unsqueeze(0).unsqueeze(0)) / (std.unsqueeze(0).unsqueeze(0) + 1e-6)
        else:             # [B,obs]
            return (x_t - mean.unsqueeze(0)) / (std.unsqueeze(0) + 1e-6)

class RecurrentPolicy(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=128):
        super().__init__()
        self.inp = nn.Linear(obs_dim, hidden)
        self.rnn = nn.GRU(hidden, hidden, batch_first=True)
        self.pi  = nn.Linear(hidden, act_dim)
        self.v   = nn.Linear(hidden, 1)
    def forward(self, x, h=None):
        z = torch.relu(self.inp(x))
        y, h2 = self.rnn(z, h)
        return self.pi(y), self.v(y).squeeze(-1), h2

@torch.no_grad()
def collect_rollout(envs, policy, T=96, device="cuda", obs_norm: ObsNormalizer=None):
    B = len(envs)
    obs_dim = len(envs[0]._make_obs())
    obs = torch.zeros(T, B, obs_dim, device=device)
    act = torch.zeros(T, B, dtype=torch.long, device=device)
    ret = torch.zeros(T, B, device=device)
    adv = torch.zeros(T, B, device=device)
    logp= torch.zeros(T, B, device=device)
    val = torch.zeros(T, B, device=device)
    mask= torch.ones (T, B, device=device)

    h = torch.zeros(1, B, policy.rnn.hidden_size, device=device)
    o = [e.reset() for e in envs]
    for t in range(T):
        ot_np = np.array(o, dtype=np.float32)                    # [B,obs]
        if obs_norm is not None: obs_norm.update(ot_np)
        ot = torch.tensor(ot_np, dtype=torch.float32, device=device).unsqueeze(1)  # [B,1,obs]
        if obs_norm is not None: ot = obs_norm.transform(ot)

        logits, v, h = policy(ot, h)                             # logits:[B,1,A], v:[B,1]
        dist = torch.distributions.Categorical(logits=logits[:,-1,:])
        a = dist.sample()                                        # [B]
        o2, r, d, _ = zip(*[envs[i].step(int(a[i].item())) for i in range(B)])

        obs[t] = ot[:, -1, :]
        act[t] = a
        ret[t] = torch.tensor(r, dtype=torch.float32, device=device)
        val[t] = v[:, -1]
        logp[t]= dist.log_prob(a)
        mask[t]= 1.0 - torch.tensor(d, dtype=torch.float32, device=device)

        o = list(o2)

    # GAE (γ=0.99, λ=0.95)
    gamma, lam = 0.99, 0.95
    gae = torch.zeros(B, device=device)
    for t in reversed(range(T)):
        next_v = val[t+1] if t+1 < T else torch.zeros_like(val[t])
        delta = ret[t] + gamma * next_v * mask[t] - val[t]
        gae = delta + gamma * lam * mask[t] * gae
        adv[t] = gae
    ret = adv + val
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    return {"obs":obs, "act":act, "ret":ret, "adv":adv, "logp":logp, "val":val, "mask":mask}

def ppo_update(policy, opt, roll, *,
               epochs=10,            # FULL: 10 (trước 8)
               clip=0.15, vf_coef=0.7, ent_coef=0.01,
               device="cuda", grad_clip=0.5,
               value_clip=True, kl_stop=0.02):   # FULL: 0.02 (trước 0.03)
    """
    PPO update (GRU train-mode + value clipping + KL early stop).
    """
    policy.train()

    obs   = roll["obs"].to(device)    # [T,B,obs]
    act   = roll["act"].to(device)    # [T,B]
    ret   = roll["ret"].to(device)    # [T,B]
    adv   = roll["adv"].to(device)    # [T,B]
    logp0 = roll["logp"].to(device)   # [T,B]
    val0  = roll["val"].to(device)    # [T,B]
    mask  = roll["mask"].to(device)   # [T,B]

    T, B, _ = obs.shape
    H0 = torch.zeros(1, B, policy.rnn.hidden_size, device=device)

    for _ in range(epochs):
        # batch_first => [B,1,obs]
        h = H0
        logits_ts, vals_ts = [], []
        for t in range(T):
            xt = obs[t].unsqueeze(1)              # [B,1,obs]  <-- FIX batch_first
            lt, vt, h = policy(xt, h)             # lt:[B,1,A], vt:[B,1]
            logits_ts.append(lt[:, -1, :])        # [B,A]
            vals_ts.append(vt[:, -1])             # [B]
        logits = torch.stack(logits_ts, 0)        # [T,B,A]
        vals   = torch.stack(vals_ts,   0)        # [T,B]

        dist = torch.distributions.Categorical(logits=logits)
        logp = dist.log_prob(act)                 # [T,B]
        ratio = torch.exp(logp - logp0)

        surr1 = ratio * adv
        surr2 = torch.clamp(ratio, 1.0 - clip, 1.0 + clip) * adv
        pol_loss = -torch.sum(torch.min(surr1, surr2) * mask) / (mask.sum() + 1e-8)

        if value_clip:
            v_clipped = val0 + (vals - val0).clamp(-clip, clip)
            v_loss1   = (vals - ret) ** 2
            v_loss2   = (v_clipped - ret) ** 2
            val_loss  = torch.sum(torch.max(v_loss1, v_loss2) * mask) / (mask.sum() + 1e-8)
        else:
            val_loss  = torch.sum((vals - ret) ** 2 * mask) / (mask.sum() + 1e-8)

        ent = torch.sum(dist.entropy() * mask) / (mask.sum() + 1e-8)
        loss = pol_loss + vf_coef * val_loss - ent_coef * ent

        approx_kl = torch.sum((logp0 - logp) * mask) / (mask.sum() + 1e-8)

        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), grad_clip)
        opt.step()

        if approx_kl.item() > kl_stop:
            break


**CELL 10**

In [10]:
# === Cell 10: Trainer (FULL; entropy chậm + PPO mới + cosine LR + quick eval) ===
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, time, math

def nll_gauss(mu, logvar, y):
    return 0.5 * (logvar + (y - mu)**2 / (logvar.exp() + 1e-6))

def _fmt_time(sec): sec=max(0,int(sec)); return f"{sec//60:02d}:{sec%60:02d}"
def _bar(p,w=12): p=float(np.clip(p,0,1)); k=int(round(p*w)); return "="*k+"-"*(w-k)

class MetaRLE2ETrainer:
    def __init__(self, data_np, day_windows, sampler,
                 device, dapm_model, anp_model,
                 demand_col=0, cov_dim=0, history_len=288,
                 lr_models=5e-4, lr_policy=1e-3):
        self.data_np = data_np; self.day_windows=day_windows; self.sampler=sampler
        self.device=device; self.demand_col=demand_col; self.cov_dim=cov_dim; self.history_len=history_len
        self.dapm_model=dapm_model.to(device); self.anp_model=anp_model.to(device)
        self.opt_dapm=torch.optim.AdamW(self.dapm_model.parameters(), lr=lr_models)
        self.opt_anp =torch.optim.AdamW(self.anp_model.parameters(),  lr=lr_models)
        self.policy=None; self.opt_policy=None
        self.obs_norm = ObsNormalizer()

    def _make_windows(self, batch_size=64):
        W,H=self.history_len,1; N=len(self.data_np)
        idxs=np.random.randint(W, max(W+1,N-1), size=batch_size)
        Xp,Up,Uf,Y=[],[],[],[]
        for i in idxs:
            past_all=self.data_np[i-W:i,:]
            if self.cov_dim>0:
                past_x=past_all[:,:-self.cov_dim]; past_u=past_all[:,-self.cov_dim:]; fut_u=self.data_np[i:i+H,-self.cov_dim:]
            else:
                past_x=past_all; past_u=np.zeros((W,0),np.float32); fut_u=np.zeros((H,0),np.float32)
            y=self.data_np[i:i+H,self.demand_col]
            Xp.append(past_x);Up.append(past_u);Uf.append(fut_u);Y.append(y)
        Xp=torch.tensor(np.stack(Xp),dtype=torch.float32,device=self.device)
        Up=torch.tensor(np.stack(Up),dtype=torch.float32,device=self.device)
        Uf=torch.tensor(np.stack(Uf),dtype=torch.float32,device=self.device)
        Y =torch.tensor(np.stack(Y), dtype=torch.float32,device=self.device).unsqueeze(-1)
        return Xp,Up,Uf,Y

    def _supervised_losses(self, Xp,Up,Uf,Y):
        mu_d,lv_d=self.dapm_model(Xp,Up,Uf); mu_a,lv_a=self.anp_model(Xp,Up,Uf)
        loss_d=nll_gauss(mu_d[:,0,:],lv_d[:,0,:],Y[:,0,:]).mean()
        loss_a=nll_gauss(mu_a[:,0,:],lv_a[:,0,:],Y[:,0,:]).mean()
        return loss_d,loss_a

    def _build_policy_if_needed(self):
        if self.policy is not None: return
        t=self.sampler.sample_one()
        env_tmp=RealEnv(self.data_np,t,max_steps=2,history_len=self.history_len,
                        demand_col=self.demand_col,cov_dim=self.cov_dim)
        obs_dim=int(env_tmp._make_obs().shape[0])
        self.policy=RecurrentPolicy(obs_dim,3,128).to(self.device)
        self.opt_policy=torch.optim.Adam(self.policy.parameters(), lr=1e-3)
        print(f"[Trainer] OBS_DIM={obs_dim} (inferred)")

    def _imitation_pretrain(self, epochs=3, T=72, bc_batch_size=256):
        self._build_policy_if_needed(); self.policy.train()
        for ep in range(epochs):
            t0=time.time()
            tasks=self.sampler.sample(8); obs_list=[]; act_list=[]
            for t in tasks:
                env=RealEnv(self.data_np,t,max_steps=T,history_len=self.history_len,
                            demand_col=self.demand_col,cov_dim=self.cov_dim)
                o=env.reset()
                for _ in range(T):
                    last_d=o[0]; th=env.tcfg.sla_cpu*100.0
                    a=2 if last_d>env.vm*th else (0 if last_d<max(1,env.vm-1)*th*0.6 else 1)
                    obs_list.append(o); act_list.append(a)
                    o,_,d,_=env.step(a)
                    if d: break
            obs_all=torch.tensor(np.array(obs_list),dtype=torch.float32,device=self.device)
            act_all=torch.tensor(np.array(act_list),dtype=torch.long,device=self.device)
            N=obs_all.size(0); perm=torch.randperm(N,device=self.device)
            loss_sum=0.0
            for st in range(0,N,bc_batch_size):
                idx=perm[st:st+bc_batch_size]; x=obs_all[idx].unsqueeze(1); y=act_all[idx]
                x = self.obs_norm.transform(x)
                B=x.size(0); h0=torch.zeros(1,B,self.policy.rnn.hidden_size,device=self.device)
                logits,_,_=self.policy(x,h0); loss=F.cross_entropy(logits[:,-1,:],y)
                self.opt_policy.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(),0.5); self.opt_policy.step()
                loss_sum+=loss.item()*B
            print(f"[BC] epoch {ep+1}/{epochs} time={_fmt_time(time.time()-t0)} loss={loss_sum/max(1,N):.4f}")
        self.policy.eval()

    def _cosine_lr(self, base, it, total):
        ratio=0.5*(1+math.cos(math.pi*it/max(1,total)))
        return base*(0.1+0.9*ratio)

    @torch.no_grad()
    def _quick_eval(self, tasks_n=5, T=64):
        tasks=self.sampler.sample(tasks_n)
        ret_list=[]; viol_cnt=0; total=0
        for t in tasks:
            env=RealEnv(self.data_np,t,max_steps=T,history_len=self.history_len,
                        demand_col=self.demand_col,cov_dim=self.cov_dim)
            o=env.reset(); h=torch.zeros(1,1,self.policy.rnn.hidden_size,device=self.device)
            ret=0.0
            for _ in range(T):
                obs=torch.tensor(o,dtype=torch.float32,device=self.device).unsqueeze(0).unsqueeze(0)
                obs=self.obs_norm.transform(obs)
                logits, v, h = self.policy(obs, h)
                a=torch.distributions.Categorical(logits=logits[:,-1,:]).sample().item()
                o, r, d, info = env.step(int(a))
                if info["cpu_util"] > env.tcfg.sla_cpu*100.0: viol_cnt += 1
                total += 1; ret += r
                if d: break
            ret_list.append(ret)
        viol_pct = 100.0 * (viol_cnt / max(1,total))
        return float(np.mean(ret_list)), viol_pct

    def train(self, total_iters=1000, meta_batch=12, T=72, K_sup=2, M_rl=1, grad_clip=0.5):
        self._build_policy_if_needed()
        self._imitation_pretrain(epochs=3, T=T, bc_batch_size=256)

        t0=time.time(); last_nl=False
        base_lr_pol=self.opt_policy.param_groups[0]["lr"]
        base_lr_d  =self.opt_dapm.param_groups[0]["lr"]
        base_lr_a  =self.opt_anp.param_groups[0]["lr"]

        for it in range(1,total_iters+1):
            # cosine LR
            lr_pol=self._cosine_lr(base_lr_pol,it,total_iters)
            lr_d  =self._cosine_lr(base_lr_d,  it,total_iters)
            lr_a  =self._cosine_lr(base_lr_a,  it,total_iters)
            for g in self.opt_policy.param_groups: g["lr"]=lr_pol
            for g in self.opt_dapm.param_groups:  g["lr"]=lr_d
            for g in self.opt_anp.param_groups:   g["lr"]=lr_a

            # ---- supervised joint ----
            self.dapm_model.train(); self.anp_model.train()
            ld=la=torch.tensor(0.0,device=self.device)
            for _ in range(K_sup):
                Xp,Up,Uf,Y=self._make_windows(batch_size=64)
                ld_,la_=self._supervised_losses(Xp,Up,Uf,Y)
                tot=ld_+la_
                self.opt_dapm.zero_grad(set_to_none=True); self.opt_anp.zero_grad(set_to_none=True)
                tot.backward()
                nn.utils.clip_grad_norm_(self.dapm_model.parameters(),grad_clip)
                nn.utils.clip_grad_norm_(self.anp_model.parameters(), grad_clip)
                self.opt_dapm.step(); self.opt_anp.step()
                ld,la=ld_,la_
            sup_loss_val=float((ld+la).item())

            # ---- PPO Meta-RL ----
            self.policy.train()
            tasks=self.sampler.sample(meta_batch)
            envs=[RealEnv(self.data_np,t,max_steps=T,history_len=self.history_len,
                          demand_col=self.demand_col,cov_dim=self.cov_dim) for t in tasks]
            roll=collect_rollout(envs,self.policy,T=T,device=self.device,obs_norm=self.obs_norm)

            # Entropy schedule chậm hơn (FULL)
            ent0, ent1 = 0.040, 0.010
            ent_coef=float(ent0+(ent1-ent0)*(it/max(1,total_iters)))

            # PPO cập nhật với tham số FULL (epochs=10, kl_stop=0.02)
            ppo_update(self.policy,self.opt_policy,roll,device=self.device,
                       ent_coef=ent_coef, epochs=10, clip=0.15, vf_coef=0.7,
                       value_clip=True, kl_stop=0.02)

            # progress
            el=time.time()-t0; prog=it/float(total_iters)
            est=el/prog if prog>0 else 0.0
            print(f"\riter {it:04d}/{total_iters}: {_bar(prog)} "
                  f"[{_fmt_time(el)} / {_fmt_time(est)}] "
                  f"sup_loss={sup_loss_val:.4f}  ent_coef={ent_coef:.4f}  lr_p={lr_pol:.2e}",
                  end="", flush=True)
            last_nl=True
            if it % 50 == 0:
                ret_mean, viol_pct = self._quick_eval(tasks_n=5, T=64)
                print(f"\n  -> quick_eval: ret_mean={ret_mean:.2f}  viol={viol_pct:.2f}%")
                last_nl=False

            if it%200==0:
                torch.save({
                    "dapm": self.dapm_model.state_dict(),
                    "anp":  self.anp_model.state_dict(),
                    "pol":  self.policy.state_dict(),
                    "it":   it
                }, f"ckpt_it{it}.pt")

        if last_nl: print()
        return self.policy

    def meta_test(self, task, episodes=2, T=128):
        env=RealEnv(self.data_np,task,max_steps=T,history_len=self.history_len,
                    demand_col=self.demand_col,cov_dim=self.cov_dim)
        rets=[]
        for ep in range(episodes):
            o=env.reset(); h=torch.zeros(1,1,self.policy.rnn.hidden_size,device=self.device)
            ret=0.0
            for _ in range(T):
                obs=torch.tensor(o,dtype=torch.float32,device=self.device).unsqueeze(0).unsqueeze(0)
                obs=self.obs_norm.transform(obs)
                with torch.no_grad():
                    logits,v,h=self.policy(obs,h)
                    a=torch.distributions.Categorical(logits=logits[:,-1,:]).sample()
                o,r,d,_=env.step(int(a.item())); ret+=r
                if d: break
            rets.append(ret)
        return rets


**CELL 11**

In [11]:
# === Cell 11: Build day_windows (adaptive) + Init & Train (gộp) ===
import numpy as np
import pandas as pd
import torch
import warnings

device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------ 1) Thích nghi history_len & build day_windows an toàn ------------

# Mục tiêu mong muốn
TARGET_HISTORY_LEN = 288          # bạn muốn 24h
HISTORY_LEN = TARGET_HISTORY_LEN  # sẽ giảm nếu dữ liệu không đủ
MIN_WIN = HISTORY_LEN + 2

STEPS_PER_DAY = 288               # 300s/step -> 288 điểm/ngày
N = len(df)
day_windows = {}

def build_by_timecol(df, min_win):
    """Dùng cột thời gian (nếu có) để gom theo ngày, trả dict {i: (s,e)}."""
    day_windows_local = {}
    time_cols = [c for c in df.columns if any(k in c.lower() for k in ["time", "date", "timestamp"])]
    if not time_cols:
        return {}
    tcol = time_cols[0]
    dt = pd.to_datetime(df[tcol], errors="coerce")
    day_id = dt.dt.normalize()
    groups = {}
    for idx, d in enumerate(day_id):
        if pd.isna(d):
            continue
        if d not in groups:
            groups[d] = [idx, idx]
        else:
            groups[d][1] = idx
    i = 1
    for d in sorted(groups.keys()):
        s, e = groups[d][0], groups[d][1] + 1
        if e - s >= min_win:
            day_windows_local[i] = (int(s), int(e))
            i += 1
    return day_windows_local

def build_by_chunks(df, min_win, chunk=STEPS_PER_DAY):
    """Không có timecol hoặc ngày quá ngắn -> chia đều theo chunk, lọc min_win."""
    N = len(df)
    day_windows_local = {}
    i = 1
    for start in range(0, N, chunk):
        end = min(start + chunk, N)
        if end - start >= min_win:
            day_windows_local[i] = (start, end)
            i += 1
    return day_windows_local

def build_by_full_series(df, min_win):
    """Fallback cuối: lấy cả chuỗi hoặc lăn cửa sổ dài min_win."""
    N = len(df)
    if N >= min_win:
        return {1: (0, N)}
    return {}

# Thử theo time column
day_windows = build_by_timecol(df, MIN_WIN)

# Nếu rỗng → chia theo block 288
if len(day_windows) == 0:
    day_windows = build_by_chunks(df, MIN_WIN, chunk=STEPS_PER_DAY)

# Nếu vẫn rỗng → cả chuỗi
if len(day_windows) == 0:
    day_windows = build_by_full_series(df, MIN_WIN)

# Nếu vẫn rỗng → giảm HISTORY_LEN rồi làm lại
if len(day_windows) == 0:
    # giảm history_len theo dữ liệu: 1/3 độ dài là “đủ an toàn”
    HISTORY_LEN = max(96, min(TARGET_HISTORY_LEN, N // 3 - 2))
    MIN_WIN = HISTORY_LEN + 2
    warnings.warn(f"Giảm history_len xuống {HISTORY_LEN} do dữ liệu ngắn (N={N}).")
    day_windows = build_by_timecol(df, MIN_WIN)
    if len(day_windows) == 0:
        day_windows = build_by_chunks(df, MIN_WIN, chunk=max(STEPS_PER_DAY, MIN_WIN))
    if len(day_windows) == 0:
        day_windows = build_by_full_series(df, MIN_WIN)

assert len(day_windows) > 0, (
    f"Không tạo được day_windows — dữ liệu quá ngắn (N={N}) "
    f"ngay cả khi giảm history_len={HISTORY_LEN}."
)

print(f"[day_windows] {len(day_windows)} cửa sổ | HISTORY_LEN={HISTORY_LEN} | ví dụ:", list(day_windows.items())[:3])

# ------------ 2) Khởi tạo model & huấn luyện (dùng HISTORY_LEN thích nghi) ------------

# Giữ thông số mô hình như trước
HORIZON = 1     # 1-step cho joint loss; RL rollout dùng T
HIDDEN  = 128

dapm_model = DAPMForecaster(
    feature_dim=X_DIM, covariate_dim=COV_DIM, hidden_dim=HIDDEN,
    output_dim=1, horizon=HORIZON, n_lstm_layers=2, dropout=0.15
).to(device)

anp_model = CPUForecaster(
    x_dim=X_DIM, cov_dim=COV_DIM, y_dim=1, hidden_dim=HIDDEN,
    horizon=HORIZON, num_heads=8, n_lstm_layers=2, dropout=0.2,
    demand_col=0
).to(device)

# End-to-end: mở gradient cả hai
for p in dapm_model.parameters(): p.requires_grad = True
for p in anp_model.parameters():  p.requires_grad = True

# Sampler & Trainer (truyền history_len=HISTORY_LEN đã thích nghi)
sampler = TaskSampler(day_windows)
trainer = MetaRLE2ETrainer(
    data_np=data_np, day_windows=day_windows, sampler=sampler,
    device=device, dapm_model=dapm_model, anp_model=anp_model,
    demand_col=DEMAND_COL_IDX, cov_dim=COV_DIM, history_len=HISTORY_LEN
)

# Huấn luyện (lịch đã nâng)
# FAST 
# policy = trainer.train(
#     total_iters=200,     # ↓
#     meta_batch=6,        # ↓
#     T=48,                # ↓
#     K_sup=1,             # ↓
#     M_rl=1,
#     grad_clip=0.5
# )
#STANDARD
# policy = trainer.train(
#     total_iters=50,    # +100 vòng cho PPO “ngấm” hơn
#     meta_batch=12,      # đa dạng hơn một chút
#     T=64,
#     K_sup=1,
#     M_rl=1,
#     grad_clip=0.5
# )

#FULL
policy = trainer.train(
    total_iters=3000,
    meta_batch=12,
    T=72,
    K_sup=2,
    M_rl=1,
    grad_clip=0.5
)



print("Trainer ready. Policy trained.")


[day_windows] 1 cửa sổ | HISTORY_LEN=288 | ví dụ: [(1, (0, 2243))]
[Trainer] OBS_DIM=9 (inferred)
[BC] epoch 1/3 time=00:08 loss=0.8593
[BC] epoch 2/3 time=00:07 loss=0.2467
[BC] epoch 3/3 time=00:07 loss=0.1302
iter 0050/3000: ------------ [10:51 / 651:15] sup_loss=0.0411  ent_coef=0.0395  lr_p=9.99e-04
  -> quick_eval: ret_mean=-30.07  viol=1.56%
iter 0100/3000: ------------ [21:45 / 652:32] sup_loss=0.4576  ent_coef=0.0390  lr_p=9.98e-04
  -> quick_eval: ret_mean=-30.18  viol=0.00%
iter 0150/3000: =----------- [32:39 / 653:10] sup_loss=0.1067  ent_coef=0.0385  lr_p=9.94e-04
  -> quick_eval: ret_mean=-19.77  viol=1.25%
iter 0200/3000: =----------- [43:34 / 653:30] sup_loss=-0.0216  ent_coef=0.0380  lr_p=9.90e-04
  -> quick_eval: ret_mean=-24.57  viol=1.56%
iter 0250/3000: =----------- [54:34 / 654:58] sup_loss=0.0317  ent_coef=0.0375  lr_p=9.85e-04
  -> quick_eval: ret_mean=-29.96  viol=0.00%
iter 0300/3000: =----------- [65:37 / 656:10] sup_loss=-0.0283  ent_coef=0.0370  lr_p=9.78e-

In [12]:
# === Cell: FINAL EVAL & REPORT (áp dụng cho FULL/STANDARD) ===
import numpy as np, torch, json, math, time

def _inv_scale(vec, scaler, key):
    if scaler is None: return np.asarray(vec, np.float32)
    m = scaler["means"][key]; s = scaler["stds"][key] if scaler["stds"][key]!=0 else 1e-6
    return np.asarray(vec, np.float32) * s + m

def _safe_mape(y_true, y_pred, eps=1e-6):
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred)/denom)) * 100.0)
def _rmse(y_true, y_pred): return float(np.sqrt(np.mean((y_true - y_pred)**2)))
def _mae(y_true, y_pred):  return float(np.mean(np.abs(y_true - y_pred)))
def _picp(y_true, y_pred, y_std, alpha=0.10):
    try:
        from scipy.stats import norm
        z = norm.ppf(1.0 - alpha/2.0)
        lo = y_pred - z * y_std
        hi = y_pred + z * y_std
        return float(np.mean((y_true >= lo) & (y_true <= hi)) * 100.0)
    except Exception:
        return None

@torch.no_grad()
def _forecast_once(past_x, past_u, fut_u, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    # DAPM
    mu_d, sd_d = dapm_model.predict(past_x, past_u, fut_u, device=device)
    # ANP
    out = anp_model.predict(past_x, past_u, fut_u, device=device)
    mu_a, sd_a = out["mean"], out["std"]
    # mix
    mean = 0.5 * (mu_d + mu_a)
    var  = 0.5 * (sd_d**2 + sd_a**2) + 1e-6
    std  = np.sqrt(var)
    return mean, std, mu_d, sd_d, mu_a, sd_a

# ---------- 1) OFFLINE FORECAST REPORT ----------
W = getattr(trainer, "history_len", 288)
N = len(data_np)
col0 = df.columns[0]

y_true_raw, y_dapm_raw, y_anp_raw, y_mix_raw = [], [], [], []
y_anp_std_raw, y_mix_std_raw = [], []
for i in range(W, N):
    past_all = data_np[i-W:i, :]
    if COV_DIM > 0:
        past_x = past_all[:, :-COV_DIM]; past_u = past_all[:, -COV_DIM:]; fut_u = data_np[i:i+1, -COV_DIM:]
    else:
        past_x = past_all; past_u = np.zeros((W,0), np.float32); fut_u = np.zeros((1,0), np.float32)
    y = data_np[i, DEMAND_COL_IDX]
    mix_mu, mix_std, mu_d, sd_d, mu_a, sd_a = _forecast_once(past_x, past_u, fut_u)
    y_true_raw.append(y)
    y_mix_raw.append(mix_mu[0,0]); y_mix_std_raw.append(mix_std[0,0])
    y_dapm_raw.append(mu_d[0,0])
    y_anp_raw.append(mu_a[0,0]); y_anp_std_raw.append(sd_a[0,0])

y_true_raw = np.array(y_true_raw, np.float32)
y_mix_raw  = np.array(y_mix_raw,  np.float32); y_mix_std_raw = np.array(y_mix_std_raw, np.float32)
y_dapm_raw = np.array(y_dapm_raw, np.float32)
y_anp_raw  = np.array(y_anp_raw,  np.float32); y_anp_std_raw = np.array(y_anp_std_raw, np.float32)

# nghịch chuẩn hoá để đọc dễ
y_true = _inv_scale(y_true_raw, SCALER, col0)
y_mix  = _inv_scale(y_mix_raw,  SCALER, col0)
y_dapm = _inv_scale(y_dapm_raw, SCALER, col0)
y_anp  = _inv_scale(y_anp_raw,  SCALER, col0)
y_anp_std = (y_anp_std_raw * (SCALER["stds"][col0] if SCALER else 1.0)) if SCALER else y_anp_std_raw
y_mix_std = (y_mix_std_raw * (SCALER["stds"][col0] if SCALER else 1.0)) if SCALER else y_mix_std_raw

offline_report = {
    "samples": int(y_true.size),
    "MIX":  {"MAE": _mae(y_true, y_mix),  "RMSE": _rmse(y_true, y_mix),  "MAPE(%)": _safe_mape(y_true, y_mix),
             "PICP@90%(%)": _picp(y_true, y_mix, y_mix_std, alpha=0.10)},
    "DAPM": {"MAE": _mae(y_true, y_dapm), "RMSE": _rmse(y_true, y_dapm), "MAPE(%)": _safe_mape(y_true, y_dapm)},
    "ANP":  {"MAE": _mae(y_true, y_anp),  "RMSE": _rmse(y_true, y_anp),  "MAPE(%)": _safe_mape(y_true, y_anp),
             "PICP@90%(%)": _picp(y_true, y_anp, y_anp_std, alpha=0.10)}
}
print("=== OFFLINE FORECAST REPORT ===")
print(json.dumps(offline_report, ensure_ascii=False, indent=2))

# Nếu muốn calibrate nhanh PICP ANP về ~90%
if offline_report["ANP"]["PICP@90%(%)"] is not None:
    target_picp = 90.0
    scale = float(np.clip(target_picp / max(1e-3, offline_report["ANP"]["PICP@90%(%)"]), 1.0, 5.0))
    setattr(anp_model, "pred_std_scale", scale)
    print(f"[Calib] anp pred_std_scale -> {scale:.3f}")

# ---------- 2) META-RL EVAL (10 tasks) ----------
@torch.no_grad()
def eval_meta_rl(trainer, tasks_n=10, T=96):
    tasks = trainer.sampler.sample(tasks_n)
    rets_ep0, rets_ep1 = [], []
    viol_cnt, steps_cnt, vm_sum, steps_vm = 0, 0, 0.0, 0
    for t in tasks:
        env = RealEnv(trainer.data_np, t, max_steps=T, history_len=trainer.history_len,
                      demand_col=trainer.demand_col, cov_dim=trainer.cov_dim)
        # ep0
        o = env.reset(); h = torch.zeros(1,1,trainer.policy.rnn.hidden_size, device=trainer.device)
        ret0 = 0.0
        for _ in range(T):
            obs = torch.tensor(o, dtype=torch.float32, device=trainer.device)[None,None,:]
            if hasattr(trainer, "obs_norm"): obs = trainer.obs_norm.transform(obs)
            logits, v, h = trainer.policy(obs, h)
            a = torch.distributions.Categorical(logits=logits[:,-1,:]).sample().item()
            o, r, d, info = env.step(int(a))
            ret0 += r
            if info["cpu_util"] > env.tcfg.sla_cpu*100.0: viol_cnt += 1
            steps_cnt += 1; vm_sum += env.vm; steps_vm += 1
            if d: break
        # ep1 (thích nghi sau 1 tập)
        o = env.reset(); h = torch.zeros(1,1,trainer.policy.rnn.hidden_size, device=trainer.device)
        ret1 = 0.0
        for _ in range(T):
            obs = torch.tensor(o, dtype=torch.float32, device=trainer.device)[None,None,:]
            if hasattr(trainer, "obs_norm"): obs = trainer.obs_norm.transform(obs)
            logits, v, h = trainer.policy(obs, h)
            a = torch.distributions.Categorical(logits=logits[:,-1,:]).sample().item()
            o, r, d, info = env.step(int(a))
            ret1 += r
            if info["cpu_util"] > env.tcfg.sla_cpu*100.0: viol_cnt += 1
            steps_cnt += 1; vm_sum += env.vm; steps_vm += 1
            if d: break
        rets_ep0.append(ret0); rets_ep1.append(ret1)
    out = {
        "tasks": tasks_n,
        "return_mean_ep0": float(np.mean(rets_ep0)),
        "return_mean_ep1": float(np.mean(rets_ep1)),
        "adaptation_delta": float(np.mean(rets_ep1)-np.mean(rets_ep0)),
        "viol_pct": float(100.0 * viol_cnt / max(1, steps_cnt)),
        "avg_vm": float(vm_sum / max(1, steps_vm)),
    }
    return out

meta_report = eval_meta_rl(trainer, tasks_n=10, T=96)
print("=== META-RL REPORT (10 tasks) ===")
print(json.dumps(meta_report, ensure_ascii=False, indent=2))

# ---------- 3) AUTO SUGGESTIONS ----------
def auto_suggest(off, meta):
    tips = []
    # mục tiêu “10/10”: DAPM MAPE ≤ 6–7%; ANP PICP ≈ 90%; viol% ≤ ~3–4%; adaptation Δ > 0; ret_ep1 ≥ baseline
    if off["DAPM"]["MAPE(%)"] > 7.0:
        tips.append("Giảm L/H hoặc tăng K_sup=3 tạm thời để DAPM giảm MAPE; lr_models=3e-4.")
    if off["ANP"]["PICP@90%(%)"] is not None and abs(off["ANP"]["PICP@90%(%)"]-90.0) > 2.0:
        tips.append("Giữ `pred_std_scale` đã calibrate; nếu lệch >±2%, tăng giảm 10–20%.")
    if meta["viol_pct"] > 4.0:
        tips.append("Giảm viol%: trong RealEnv.step() tăng r_over từ -6.0 → -7.0 hoặc hạ base_tol 8.0 → 7.5.")
    if meta["return_mean_ep1"] < meta["return_mean_ep0"]:
        tips.append("Return chưa vượt sau thích nghi: giảm risk_pen scale 10–15% hoặc tăng ent_coef đầu kỳ.")
    if meta["adaptation_delta"] < 10.0:
        tips.append("Tăng epochs=10 trong ppo_update hoặc tăng meta_batch thêm 2, giữ T=72.")
    return tips

print("=== SUGGESTIONS ===")
print("\n".join(auto_suggest(offline_report, meta_report)) or "OK: Các mục tiêu đang đạt/tiệm cận.")


=== OFFLINE FORECAST REPORT ===
{
  "samples": 1955,
  "MIX": {
    "MAE": 2.2914481163024902,
    "RMSE": 2.987187147140503,
    "MAPE(%)": 5.672705918550491,
    "PICP@90%(%)": 100.0
  },
  "DAPM": {
    "MAE": 0.013545122928917408,
    "RMSE": 0.059355251491069794,
    "MAPE(%)": 0.0340841303113848
  },
  "ANP": {
    "MAE": 4.58110237121582,
    "RMSE": 5.97300910949707,
    "MAPE(%)": 11.340415477752686,
    "PICP@90%(%)": 87.62148337595907
  }
}
[Calib] anp pred_std_scale -> 1.027
=== META-RL REPORT (10 tasks) ===
{
  "tasks": 10,
  "return_mean_ep0": -27.24074740433475,
  "return_mean_ep1": -33.20216087779869,
  "adaptation_delta": -5.961413473463939,
  "viol_pct": 3.4895833333333335,
  "avg_vm": 1.8
}
=== SUGGESTIONS ===
Giữ `pred_std_scale` đã calibrate; nếu lệch >±2%, tăng giảm 10–20%.
Return chưa vượt sau thích nghi: giảm risk_pen scale 10–15% hoặc tăng ent_coef đầu kỳ.
Tăng epochs=10 trong ppo_update hoặc tăng meta_batch thêm 2, giữ T=72.


**CELL 12**

In [13]:
sampler = TaskSampler(day_windows)
rets = trainer.meta_test(sampler.sample_one(), episodes=2, T=128)
print("Meta-test episodic returns (ep0, ep1):", rets)


Meta-test episodic returns (ep0, ep1): [-60.644397618803445, -65.66214462286466]


**CELL 13**

In [14]:
# OBS_DIM = trainer.policy.inp.in_features
# app_py = f'''
# from fastapi import FastAPI
# import torch
# from torch import nn
# from pydantic import BaseModel
# from typing import List

# class RecurrentPolicy(nn.Module):
#     def __init__(self, obs_dim:int, act_dim:int=3, hidden:int=128):
#         super().__init__()
#         self.inp=nn.Linear(obs_dim, hidden)
#         self.rnn=nn.GRU(hidden, hidden, batch_first=True)
#         self.pi=nn.Linear(hidden, act_dim)
#         self.v =nn.Linear(hidden, 1)
#     def forward(self, x, h=None):
#         z=torch.relu(self.inp(x)); y,h2=self.rnn(z,h); return self.pi(y), self.v(y).squeeze(-1), h2

# policy=RecurrentPolicy({OBS_DIM}, 3, 128)
# policy.load_state_dict(torch.load("meta_rl_policy_gru.pt", map_location="cpu"))
# policy.eval()

# class Obs(BaseModel):
#     features: List[float]

# app = FastAPI()

# @app.post("/decide")
# def decide(obs: Obs):
#     with torch.no_grad():
#         x=torch.tensor([obs.features], dtype=torch.float32).unsqueeze(1)  # [1,1,OBS_DIM]
#         logits, v, _ = policy(x)
#         a = torch.distributions.Categorical(logits=logits[:,-1,:]).probs.argmax(dim=-1).item()
#     return {{"action": int(a)}}
# '''
# with open("app.py","w",encoding="utf-8") as f: f.write(app_py)

# dockerfile = '''
# FROM python:3.10-slim
# WORKDIR /app
# COPY meta_rl_policy_gru.pt /app/
# COPY app.py /app/
# RUN pip install fastapi uvicorn torch --no-cache-dir
# EXPOSE 8000
# CMD ["uvicorn","app:app","--host","0.0.0.0","--port","8000"]
# '''
# with open("Dockerfile","w",encoding="utf-8") as f: f.write(dockerfile)

# print("Generated app.py & Dockerfile — sẵn sàng build & deploy.")


In [15]:
import numpy as np

def inv_scale(vec, scaler, key):
    """vec: (...,) hoặc (...,1) ở scale z-score; trả về đơn vị gốc theo cột `key`."""
    if scaler is None: return vec
    m = scaler["means"][key]; s = scaler["stds"][key] if scaler["stds"][key]!=0 else 1e-6
    return vec * s + m

def safe_mape(y_true, y_pred, eps=1e-6):
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred)/denom)) * 100.0)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred)**2)))

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def picp(y_true, y_pred, y_std, alpha=0.1):
    """Prediction Interval Coverage Probability ~ (1-alpha) cho Gaussian (µ±z*σ)."""
    from scipy.stats import norm
    z = norm.ppf(1.0 - alpha/2.0)
    lo = y_pred - z * y_std
    hi = y_pred + z * y_std
    inside = (y_true >= lo) & (y_true <= hi)
    return float(np.mean(inside) * 100.0)


**CELL 14**

In [16]:
# === Cell #14: Offline forecast report (rolling) + calibration nhẹ ===
import numpy as np, json

def inv_scale(vec, scaler, key):
    if scaler is None: return vec
    m = scaler["means"][key]; s = scaler["stds"][key] if scaler["stds"][key]!=0 else 1e-6
    return vec * s + m

def safe_mape(y_true, y_pred, eps=1e-6):
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred)/denom)) * 100.0)
def rmse(y_true, y_pred): return float(np.sqrt(np.mean((y_true - y_pred)**2)))
def mae(y_true, y_pred):  return float(np.mean(np.abs(y_true - y_pred)))

def picp(y_true, y_pred, y_std, alpha=0.1):
    try:
        from scipy.stats import norm
        z = norm.ppf(1.0 - alpha/2.0)
        lo = y_pred - z * y_std
        hi = y_pred + z * y_std
        return float(np.mean((y_true >= lo) & (y_true <= hi)) * 100.0)
    except Exception:
        return None

W = 288
N = len(data_np)
col0 = df.columns[0]

y_true_raw, y_dapm_raw, y_anp_raw, y_anp_std_raw = [], [], [], []

for i in range(W, N):
    past_all = data_np[i-W:i, :]
    if COV_DIM > 0:
        past_x = past_all[:, :-COV_DIM]
        past_u = past_all[:, -COV_DIM:]
        fut_u  = data_np[i:i+1, -COV_DIM:]
    else:
        past_x = past_all
        past_u = np.zeros((W,0), np.float32)
        fut_u  = np.zeros((1,0), np.float32)

    y = data_np[i, DEMAND_COL_IDX]

    # DAPM
    mu_d, sd_d = dapm_model.predict(past_x, past_u, fut_u, device=device)  # (H,1)
    # ANP
    out = anp_model.predict(past_x, past_u, fut_u, device=device)          # dict
    mu_a, sd_a = out["mean"], out["std"]

    # dùng mean ANP để report ANP; DAPM riêng
    y_true_raw.append(y)
    y_dapm_raw.append(mu_d[0,0])
    y_anp_raw.append(mu_a[0,0])
    y_anp_std_raw.append(sd_a[0,0])

y_true_raw = np.asarray(y_true_raw, np.float32)
y_dapm_raw = np.asarray(y_dapm_raw, np.float32)
y_anp_raw  = np.asarray(y_anp_raw,  np.float32)
y_anp_std_raw = np.asarray(y_anp_std_raw, np.float32)

# nghịch chuẩn hoá (đơn vị %CPU thật nếu SCALER là của cột demand)
y_true = inv_scale(y_true_raw, SCALER, col0)
y_dapm = inv_scale(y_dapm_raw, SCALER, col0)
y_anp  = inv_scale(y_anp_raw,  SCALER, col0)
y_anp_std = (y_anp_std_raw * (SCALER["stds"][col0] if SCALER else 1.0)) if SCALER else y_anp_std_raw

offline_report = {
    "samples": int(y_true.size),
    "DAPM": {
        "MAE": mae(y_true, y_dapm),
        "RMSE": rmse(y_true, y_dapm),
        "MAPE(%)": safe_mape(y_true, y_dapm),
    },
    "ANP":  {
        "MAE": mae(y_true, y_anp),
        "RMSE": rmse(y_true, y_anp),
        "MAPE(%)": safe_mape(y_true, y_anp),
        "PICP@90%(%)": picp(y_true, y_anp, y_anp_std, alpha=0.10),
    }
}
print("=== OFFLINE FORECAST REPORT (full-series rolling) ===")
print(json.dumps(offline_report, ensure_ascii=False, indent=2))

# Calibration nhẹ cho ANP (đưa PICP gần 90%)
target_picp = 90.0
cur_picp = float(offline_report["ANP"]["PICP@90%(%)"]) if offline_report["ANP"]["PICP@90%(%)"] is not None else 0.0
scale = target_picp / max(1e-3, cur_picp)
scale = float(np.clip(scale, 0.8, 1.5))   # biên nhỏ vì ANP đã học logvar
setattr(anp_model, "pred_std_scale", scale)
print("ANP pred_std_scale set to:", getattr(anp_model, "pred_std_scale"))

# (Tuỳ chọn) báo lại PICP sau scale
y_anp_std_scaled = y_anp_std * getattr(anp_model, "pred_std_scale")
offline_report_after = {
    "samples": int(y_true.size),
    "DAPM": offline_report["DAPM"],
    "ANP_after_scale": {
        "MAE": offline_report["ANP"]["MAE"],
        "RMSE": offline_report["ANP"]["RMSE"],
        "MAPE(%)": offline_report["ANP"]["MAPE(%)"],
        "PICP@90%(%)": picp(y_true, y_anp, y_anp_std_scaled, alpha=0.10),
        "std_scale": getattr(anp_model, "pred_std_scale"),
    }
}
print("=== OFFLINE FORECAST REPORT (after ANP std scaling) ===")
print(json.dumps(offline_report_after, ensure_ascii=False, indent=2))


=== OFFLINE FORECAST REPORT (full-series rolling) ===
{
  "samples": 1955,
  "DAPM": {
    "MAE": 0.013545122928917408,
    "RMSE": 0.059355251491069794,
    "MAPE(%)": 0.0340841303113848
  },
  "ANP": {
    "MAE": 4.58110237121582,
    "RMSE": 5.97300910949707,
    "MAPE(%)": 11.340415477752686,
    "PICP@90%(%)": 89.05370843989769
  }
}
ANP pred_std_scale set to: 1.0106260769672604
=== OFFLINE FORECAST REPORT (after ANP std scaling) ===
{
  "samples": 1955,
  "DAPM": {
    "MAE": 0.013545122928917408,
    "RMSE": 0.059355251491069794,
    "MAPE(%)": 0.0340841303113848
  },
  "ANP_after_scale": {
    "MAE": 4.58110237121582,
    "RMSE": 5.97300910949707,
    "MAPE(%)": 11.340415477752686,
    "PICP@90%(%)": 89.61636828644501,
    "std_scale": 1.0106260769672604
  }
}


**CELL 15**

In [17]:
def run_episode(env, policy, greedy=False, T=128, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    obs = torch.tensor(env.reset(), dtype=torch.float32, device=device)[None,None,:]
    h = torch.zeros(1,1,policy.rnn.hidden_size, device=device)
    ret=0.0; sla_viol=0; steps=0; vm_sum=0
    for _ in range(T):
        if greedy:
            with torch.no_grad():
                logits, v, h = policy(obs, h)
                a = torch.distributions.Categorical(logits=logits[:,-1,:]).probs.argmax(dim=-1)
        else:
            with torch.no_grad():
                logits, v, h = policy(obs, h)
                a = torch.distributions.Categorical(logits=logits[:,-1,:]).sample()
        o2, r, d, info = env.step(int(a.item()))
        ret += r; steps += 1; vm_sum += info["vm"]
        # 1 nếu vượt SLA
        # cpu_util%/100 > sla_cpu → vi phạm
        if (info["cpu_util"]/100.0) > env.tcfg.sla_cpu: sla_viol += 1
        obs = torch.tensor(o2, dtype=torch.float32, device=device)[None,None,:]
        if d: break
    return {
        "return": float(ret),
        "viol_rate(%)": float(sla_viol/max(1,steps) * 100.0),
        "avg_vm": float(vm_sum/max(1,steps))
    }

def rule_based_baseline(env, T=128):
    """Baseline đơn giản: nếu last_demand > vm * threshold thì scale+, nếu thấp quá thì scale-."""
    obs = env.reset()
    ret=0.0; sla_viol=0; steps=0; vm_sum=0
    threshold = env.tcfg.sla_cpu * 100.0  # vì last_d (đơn vị sau scale của cột demand, ở đây coi như proxy %CPU)
    for _ in range(T):
        last_d = obs[0]
        if last_d > (env.vm * threshold): action = 2  # scale+
        elif last_d < max(1, env.vm-1) * threshold * 0.6: action = 0  # scale-
        else: action = 1
        o2, r, d, info = env.step(action)
        ret += r; steps += 1; vm_sum += info["vm"]
        if (info["cpu_util"]/100.0) > env.tcfg.sla_cpu: sla_viol += 1
        obs = o2
        if d: break
    return {
        "return": float(ret),
        "viol_rate(%)": float(sla_viol/max(1,steps) * 100.0),
        "avg_vm": float(vm_sum/max(1,steps))
    }

# Tập task kiểm thử (không trùng seed sampler.train)
# Tập task kiểm thử (không trùng seed với trainer)
rng = np.random.default_rng(777)
test_tasks = []
for _ in range(10):
    t = TaskConfig(
        sla_cpu=float(rng.choice([0.6, 0.7, 0.8])),
        vm_min=int(rng.choice([1, 2, 3])),
        vm_max=int(rng.choice([8, 10, 12])),
        cost_per_vm=float(rng.choice([0.2, 0.3, 0.4])),     # <- dùng rng, không phải self.rng
        spike_amp=float(rng.choice([1.0, 1.2, 1.4, 1.6])),  # <- dùng rng
        window=day_windows[int(rng.choice(list(day_windows.keys())))]
    )
    test_tasks.append(t)


rl_report = {"policy_sample": [], "baseline": []}
for t in test_tasks:
    env = RealEnv(data_np, t, max_steps=128, history_len=288, demand_col=DEMAND_COL_IDX, cov_dim=COV_DIM)
    # chính sách đã train
    r1 = run_episode(env, trainer.policy, greedy=False, T=128, device=device)
    # greedy (không stochastic) — để xem upper bound của chính sách hiện tại
    env2 = RealEnv(data_np, t, max_steps=128, history_len=288, demand_col=DEMAND_COL_IDX, cov_dim=COV_DIM)
    r2 = run_episode(env2, trainer.policy, greedy=True, T=128, device=device)
    # baseline rule
    env3 = RealEnv(data_np, t, max_steps=128, history_len=288, demand_col=DEMAND_COL_IDX, cov_dim=COV_DIM)
    r3 = rule_based_baseline(env3, T=128)
    rl_report["policy_sample"].append({"stoch": r1, "greedy": r2})
    rl_report["baseline"].append(r3)

def mean_std(vals): 
    a=np.array(vals, np.float32)
    return float(a.mean()), float(a.std())

pol_ret = [x["stoch"]["return"] for x in rl_report["policy_sample"]]
pol_vio = [x["stoch"]["viol_rate(%)"] for x in rl_report["policy_sample"]]
pol_vm  = [x["stoch"]["avg_vm"] for x in rl_report["policy_sample"]]

base_ret = [x["return"] for x in rl_report["baseline"]]
base_vio = [x["viol_rate(%)"] for x in rl_report["baseline"]]
base_vm  = [x["avg_vm"] for x in rl_report["baseline"]]

rl_summary = {
    "Policy (stochastic)": {
        "return_mean±std": mean_std(pol_ret),
        "viol%_mean±std":  mean_std(pol_vio),
        "avg_vm_mean±std": mean_std(pol_vm),
    },
    "Rule-based baseline": {
        "return_mean±std": mean_std(base_ret),
        "viol%_mean±std":  mean_std(base_vio),
        "avg_vm_mean±std": mean_std(base_vm),
    }
}
print("=== RL / META-RL REPORT (10 tasks) ===")
print(json.dumps(rl_summary, ensure_ascii=False, indent=2))


=== RL / META-RL REPORT (10 tasks) ===
{
  "Policy (stochastic)": {
    "return_mean±std": [
      -55.2520637512207,
      18.679349899291992
    ],
    "viol%_mean±std": [
      6.171875,
      15.239181518554688
    ],
    "avg_vm_mean±std": [
      1.7000000476837158,
      0.4582575559616089
    ]
  },
  "Rule-based baseline": {
    "return_mean±std": [
      -55.2520637512207,
      18.679349899291992
    ],
    "viol%_mean±std": [
      6.171875,
      15.239181518554688
    ],
    "avg_vm_mean±std": [
      1.7000000476837158,
      0.4582575559616089
    ]
  }
}


**CELL 16**

In [18]:
# Chạy 2 episode liên tiếp trên cùng 1 task xem phần thưởng có tăng không (adaptation)
t = TaskConfig(
    sla_cpu=0.7, vm_min=1, vm_max=10, cost_per_vm=1.0, spike_amp=1.2,
    window=day_windows[max(day_windows.keys())]
)
rets = trainer.meta_test(t, episodes=2, T=128)
meta_adapt = {"ep0_return": float(rets[0]), "ep1_return": float(rets[1]), "delta": float(rets[1]-rets[0])}
print("=== META-ADAPTATION CHECK ===")
print(json.dumps(meta_adapt, ensure_ascii=False, indent=2))


=== META-ADAPTATION CHECK ===
{
  "ep0_return": -26.64279099019068,
  "ep1_return": 39.40934796329605,
  "delta": 66.05213895348673
}
